# CN-Star Detection: delta_raw + Selective PCA

**Motivation**: The `delta_raw` config (delta_CN3839, delta_CN4142, delta_CH4300, CN3839, CN4142, CH4300) showed the best balance in the feature ablation study. However, we're only using 3800-5000A spectral range, missing information from 5000-8000A. Adding selective PCA components can recover broader spectral morphology without reintroducing strong parameter bias.

**Experiment**: Test `delta_raw` + varying numbers of PCA components (0, 3, 5, 8, 10, 15).

**Key visualization**: For each top candidate, compare its spectrum against its **cluster mean spectrum** to visually confirm CN band enhancement.

## 1. Setup

In [ ]:
import sys, os, pickle, time, warnings
warnings.filterwarnings('ignore')
from pathlib import Path

_p = Path.cwd().resolve()
for _ in range(5):
    if (_p / 'Deep' / 'data.py').exists():
        break
    _p = _p.parent
PROJECT_ROOT = _p
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({'figure.dpi': 110, 'font.size': 10})
from scipy.stats import spearmanr

import torch
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

from BinaryClassifier import augment_spectra, BinaryTrainer
from BinaryClassifier.pipeline import slice_wave_range, WAVE_START, WAVE_END
from Deep.data import (
    SpectraDataset, load_spectra_data,
    compute_physics_features_masked_cluster, compute_physics_features,
    compute_band_index, CN_BAND_DEFS,
)
from Deep.augmentation import load_gcs_spectra
from Deep.spectra_io import label_stars_by_cn_catalog

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE} | Project: {PROJECT_ROOT}')

## 2. Data Loading

In [ ]:
print('Loading LAMOST spectra...')
common_wave_full = np.arange(3710.0, 8890.0, 1.0)

dataset_full = load_spectra_data(
    stars_csv=str(PROJECT_ROOT / 'stars.csv'),
    spectra_folder=str(PROJECT_ROOT / 'dr13_new'),
    cn_catalogs=[str(PROJECT_ROOT / 'FT_cands.csv')],
    common_wave=common_wave_full, seg_width=200.0, verbose=False,
)

X_sliced, wave = slice_wave_range(dataset_full.X, common_wave_full, WAVE_START, WAVE_END)
dataset = SpectraDataset(X=X_sliced, y=dataset_full.y.copy(), meta=dataset_full.meta.copy(), wave=wave)
print(f'Dataset: {dataset.X.shape[0]} spectra x {dataset.X.shape[1]} pixels | FT_cands: {dataset.n_positive}')

In [ ]:
# Compute physics features (n_pca=15 for full flexibility)
print('Computing physics features + masked-band clustering...')
phys_result = compute_physics_features_masked_cluster(
    dataset, n_clusters=45, k_neighbors=180, n_pca=3, random_seed=42,
)
dataset_phys_base = phys_result['dataset']  # 12-D
cluster_labels = phys_result['cluster_labels']
n_clusters = phys_result['n_clusters']

# Add extra PCA components (beyond 3, up to 15)
pca_full = PCA(n_components=15, random_state=42).fit(dataset.X)
pca_all = pca_full.transform(dataset.X)

# Build 24-D raw features: 9 base + 15 PCA
X_base9 = dataset_phys_base.X[:, :9]  # teff,logg,feh,CN3,delta3
X_raw_24d = np.column_stack([X_base9, pca_all]).astype(np.float32)
scaler_24d = StandardScaler().fit(X_raw_24d)
X_24d_std = scaler_24d.transform(X_raw_24d).astype(np.float32)

# Meta with physics features
meta_phys = dataset.meta.copy()
for col in ['CN3839','CN4142','CH4300','delta_CN3839','delta_CN4142','delta_CH4300']:
    if col in dataset_phys_base.meta.columns:
        meta_phys[col] = dataset_phys_base.meta[col].values
for i in range(15):
    meta_phys[f'pca_{i+1}'] = X_24d_std[:, 9 + i]

dataset_phys = SpectraDataset(X=X_24d_std, y=dataset.y.copy(), meta=meta_phys, wave=np.arange(24, dtype=float))
print(f'Physics features: {dataset_phys.X.shape} | Clusters: {n_clusters}')
print(f'PCA variance (15 comps): {pca_full.explained_variance_ratio_.sum():.2%}')

In [ ]:
# Validation sets
print('Building validation sets...')

cnstar_labeled, _ = label_stars_by_cn_catalog(
    stars_df=dataset.meta.copy(), cn_catalogs=[str(PROJECT_ROOT / 'CNstar.csv')],
    tolerance_arcsec=1.0, positive_label=1, unlabeled_value=-1,
)
cnstar_mask = cnstar_labeled['label'].values == 1

# GCS
gcs_raw, _ = load_gcs_spectra(str(PROJECT_ROOT / 'GCS'), common_wave_full)
from Deep.data import normalize_datacube_segment
gcs_norm = normalize_datacube_segment(gcs_raw, common_wave_full, seg_width=200.0, deg=2, show_progress=False)
X_gcs_sliced, _ = slice_wave_range(gcs_norm, common_wave_full, WAVE_START, WAVE_END)
gcs_cat = pd.read_csv(str(PROJECT_ROOT / 'GCS.csv'))
n_gcs = len(X_gcs_sliced)
gcs_ds = SpectraDataset(X=X_gcs_sliced, y=np.ones(n_gcs),
    meta=pd.DataFrame({'label':[1]*n_gcs, 'teff':gcs_cat['Teff'].values[:n_gcs],
                       'logg':gcs_cat['logg'].values[:n_gcs], 'feh':gcs_cat['[Fe/H]'].values[:n_gcs]}),
    wave=wave)
gcs_phys_base = compute_physics_features(gcs_ds, n_pca=3, random_seed=42)
pca_gcs = pca_full.transform(X_gcs_sliced)
gcs_raw_24d = np.column_stack([gcs_phys_base.X[:, :9], pca_gcs]).astype(np.float32)
gcs_24d_std = scaler_24d.transform(gcs_raw_24d).astype(np.float32)
gcs_phys_full = SpectraDataset(X=gcs_24d_std, y=np.ones(n_gcs), meta=gcs_phys_base.meta, wave=np.arange(24, dtype=float))

X_cnstar_phys = dataset_phys.X[cnstar_mask]
print(f'CNstar: {cnstar_mask.sum()} | GCS: {gcs_phys_full.X.shape[0]}')

## 3. Feature Configurations

All configs share the `delta_raw` core (6D): delta_CN3839, delta_CN4142, delta_CH4300, CN3839, CN4142, CH4300.
Varying: number of PCA components added.

In [ ]:
# Feature pool indices in 24-D array
# 0:teff, 1:logg, 2:feh, 3:CN3839, 4:CN4142, 5:CH4300, 6:dCN3839, 7:dCN4142, 8:dCH4300, 9..23:pca_1..pca_15
IDX_DELTA = [6, 7, 8]        # delta_CN3839, delta_CN4142, delta_CH4300
IDX_CN_RAW = [3, 4, 5]       # CN3839, CN4142, CH4300
IDX_PCA_START = 9

PCA_CONFIGS = {
    'delta_pca0': 0,   # pure delta_raw (6D)
    'delta_pca3': 3,   # + pca_1-3  (9D)
    'delta_pca5': 5,   # + pca_1-5  (11D)
    'delta_pca8': 8,   # + pca_1-8  (14D)
    'delta_pca10': 10, # + pca_1-10 (16D)
    'delta_pca15': 15, # + pca_1-15 (21D)
}

def build_config(X_24d, n_pca):
    """Build delta_raw + n PCA components from 24-D standardized data."""
    idx = IDX_DELTA + IDX_CN_RAW
    if n_pca > 0:
        idx += [IDX_PCA_START + i for i in range(n_pca)]
    X_sub = X_24d[:, idx].astype(np.float32)
    return StandardScaler().fit_transform(X_sub).astype(np.float32)

configs = {}
for name, n_pca in PCA_CONFIGS.items():
    dim = 6 + n_pca
    configs[name] = {
        'X': build_config(dataset_phys.X, n_pca),
        'dim': dim, 'n_pca': n_pca, 'name': name,
    }
    print(f'  {name:15s}: {dim:3d}D (delta_raw + {n_pca:2d} PCA)')

## 4. Train/Val Split

In [ ]:
rng = np.random.RandomState(42)

exclude_mask = np.zeros(len(dataset_phys), dtype=bool)
exclude_mask[dataset_phys.y == 1] = True  # FT_cands
exclude_mask[cnstar_mask] = True           # CNstar

neg_clusters = cluster_labels[~exclude_mask]
neg_meta_raw = dataset_phys.meta.iloc[~exclude_mask].reset_index(drop=True)
unl_indices = np.where(~exclude_mask)[0]  # original indices for spectrum lookup

def extract_subset(X_24d, indices, scaler):
    """Extract same feature subset from any 24-D array."""
    X_sub = X_24d[:, indices].astype(np.float32)
    return scaler.transform(X_sub).astype(np.float32)

config_splits = {}
for cfg_name, cfg in configs.items():
    # Build indices for this config
    idx = IDX_DELTA + IDX_CN_RAW
    if cfg['n_pca'] > 0:
        idx += [IDX_PCA_START + i for i in range(cfg['n_pca'])]
    
    scaler = StandardScaler()
    scaler.fit(cfg['X'])
    
    X_ft = scaler.transform(cfg['X'][dataset_phys.y == 1]).astype(np.float32)
    neg_pool = scaler.transform(cfg['X'][~exclude_mask]).astype(np.float32)
    X_cnstar_cfg = extract_subset(X_cnstar_phys, idx, scaler)
    X_gcs_cfg = extract_subset(gcs_phys_full.X, idx, scaler)
    
    # Augment
    X_ft_aug, _ = augment_spectra(
        X_ft, np.arange(cfg['dim'], dtype=float),
        n_noise=12, n_rv=6, n_tilt=4, n_mixup=8, n_depth=6,
        noise_std=0.0025, random_seed=42,
    )
    
    n_val_pos = max(int(len(X_ft_aug) * 0.1), 5)
    val_pos_idx = rng.choice(len(X_ft_aug), n_val_pos, replace=False)
    train_mask = np.ones(len(X_ft_aug), dtype=bool)
    train_mask[val_pos_idx] = False
    val_neg_idx = rng.choice(len(neg_pool), n_val_pos, replace=False)
    
    config_splits[cfg_name] = {
        'X_train_pos': X_ft_aug[train_mask], 'neg_pool': neg_pool,
        'X_val': np.vstack([X_ft_aug[val_pos_idx], neg_pool[val_neg_idx]]),
        'y_val': np.concatenate([np.ones(n_val_pos), np.zeros(n_val_pos)]).astype(np.float32),
        'X_cnstar': X_cnstar_cfg, 'X_gcs': X_gcs_cfg,
    }

print('Train/val splits prepared.')
for n, s in config_splits.items():
    print(f'  {n:15s}: pos={s["X_train_pos"].shape}  cnstar={s["X_cnstar"].shape}')

## 5. Training

In [ ]:
def train_ensemble(cfg_name, cfg, split, n_models=5, base_seed=42, device='cpu', verbose=True):
    rng = np.random.RandomState(base_seed)
    models = []
    n_neg = len(split['X_train_pos'])
    for i in range(n_models):
        seed = base_seed + i * 100
        neg_idx = rng.choice(len(split['neg_pool']), n_neg, replace=False)
        X_neg = split['neg_pool'][neg_idx]
        if verbose:
            print(f'  [{cfg_name}] M{i+1}/5 (seed={seed}) ', end='', flush=True)
        trainer = BinaryTrainer(
            input_dim=cfg['dim'], latent_dim=64, dropout=0.35, encoder_type='mlp',
            learning_rate=1e-4, weight_decay=1e-5, pos_weight=1.0, mixup_alpha=0.2,
            n_epochs=80, early_stopping_patience=15, random_seed=seed,
            device=device, save_dir='BinaryClassifier/checkpoints',
        )
        model = trainer.train(split['X_train_pos'], X_neg, split['X_val'], split['y_val'], verbose=False)
        models.append(model)
        best = min((h['val_loss'] for h in trainer.history if 'val_loss' in h), default=float('inf'))
        if verbose:
            print(f'done (loss={best:.4f})')
    return models

def ensemble_predict(models, X, batch_size=512, device='cpu'):
    probs = np.array([m.predict(X, batch_size, device) for m in models])
    return probs.mean(axis=0), probs.std(axis=0)

def cluster_z_score(probs, cluster_labels):
    z = np.zeros_like(probs)
    for cid in np.unique(cluster_labels):
        mask = cluster_labels == cid
        c_p = probs[mask]
        mu, sig = c_p.mean(), c_p.std()
        if sig > 1e-8 and len(c_p) >= 5:
            z[mask] = (c_p - mu) / sig
    return z

In [ ]:
all_models = {}
for cfg_name in configs.keys():
    print(f'\n{"="*60}')
    print(f'Training: {cfg_name} ({configs[cfg_name]["dim"]}D)')
    print(f'{"="*60}')
    t0 = time.time()
    all_models[cfg_name] = train_ensemble(cfg_name, configs[cfg_name], config_splits[cfg_name],
                                          device=DEVICE, verbose=True)
    print(f'  Time: {time.time()-t0:.1f}s')

## 6. Evaluation

In [ ]:
all_results = {}

for cfg_name in configs.keys():
    models = all_models[cfg_name]
    split = config_splits[cfg_name]
    
    r = {'cfg_name': cfg_name, 'dim': configs[cfg_name]['dim']}
    
    # Unlabeled predictions + z-score
    probs_unl, stds_unl = ensemble_predict(models, split['neg_pool'], 512, DEVICE)
    z_unl = cluster_z_score(probs_unl, neg_clusters)
    
    # CNstar
    probs_cn, _ = ensemble_predict(models, split['X_cnstar'], 512, DEVICE)
    r['cnstar_mean'] = float(probs_cn.mean())
    r['cnstar_median'] = float(np.median(probs_cn))
    r['cnstar_p90'] = float(np.percentile(probs_cn, 90))
    
    # GCS
    probs_gcs, _ = ensemble_predict(models, split['X_gcs'], 512, DEVICE)
    r['gcs_mean'] = float(probs_gcs.mean())
    
    # Bias (raw units)
    top200_idx = np.argsort(z_unl)[-200:][::-1]
    bias = {}
    for col in ['teff', 'logg', 'feh']:
        tv = pd.to_numeric(neg_meta_raw[col].iloc[top200_idx], errors='coerce').dropna()
        av = pd.to_numeric(neg_meta_raw[col], errors='coerce').dropna()
        bias[col] = {'shift': float(tv.median() - av.median()), 'd': float((tv.mean()-av.mean())/(av.std()+1e-8))}
    r['bias'] = bias
    
    # Within-cluster correlation
    wc_corrs, wc_weights = [], []
    delta_vals = neg_meta_raw['delta_CN3839'].values
    for cid in np.unique(neg_clusters):
        mask = neg_clusters == cid
        if mask.sum() >= 10:
            valid = ~(np.isnan(z_unl[mask]) | np.isnan(delta_vals[mask]))
            if valid.sum() >= 10:
                rho, _ = spearmanr(z_unl[mask][valid], delta_vals[mask][valid])
                wc_corrs.append(rho)
                wc_weights.append(mask.sum())
    r['wc_r_weighted'] = float(np.average(wc_corrs, weights=wc_weights)) if wc_corrs else 0
    
    # Top200 cluster diversity
    top200_clusters = neg_clusters[top200_idx]
    r['top200_n_clusters'] = len(np.unique(top200_clusters))
    
    r['_probs'] = probs_unl
    r['_z'] = z_unl
    r['_top200_idx'] = top200_idx
    
    all_results[cfg_name] = r

## 7. Results Summary

In [ ]:
print('=' * 120)
print(f'{"Config":<15} {"Dim":>4} {"CNstar":>8} {"CNstar P90":>10} {"GCS":>8} {"Teff shift":>10} {"|d|_avg":>8} {"WC r":>7} {"Clusters":>8}')
print('-' * 120)

for cfg_name in PCA_CONFIGS.keys():
    r = all_results[cfg_name]
    avg_d = np.mean([abs(r['bias'][p]['d']) for p in ['teff', 'logg', 'feh']])
    teff_shift = r['bias']['teff']['shift']
    print(f'{cfg_name:<15} {r["dim"]:>4} {r["cnstar_mean"]:>8.3f} {r["cnstar_p90"]:>10.3f} '
          f'{r["gcs_mean"]:>8.3f} {teff_shift:>+9.0f}K {avg_d:>8.3f} {r["wc_r_weighted"]:>7.3f} {r["top200_n_clusters"]:>8}')
print('=' * 120)

In [ ]:
# ---- Visual: z-score vs Teff scatter for all configs ----
config_names = list(PCA_CONFIGS.keys())
fig, axes = plt.subplots(2, 3, figsize=(22, 13))
teff_vals_all = pd.to_numeric(neg_meta_raw['teff'], errors='coerce').values

for ax, cfg_name in zip(axes.flat, config_names):
    r = all_results[cfg_name]
    valid = ~np.isnan(teff_vals_all) & ~np.isnan(r['_z'])
    ax.scatter(teff_vals_all[valid], r['_z'][valid], c='lightgray', s=1, alpha=0.3, rasterized=True)
    top200 = r['_top200_idx']
    ax.scatter(teff_vals_all[top200], r['_z'][top200], c='crimson', s=8, alpha=0.7, edgecolors='white', linewidth=0.3)
    
    # Trend line
    bins = np.linspace(teff_vals_all[valid].min(), teff_vals_all[valid].max(), 20)
    bin_meds = [np.median(r['_z'][valid & (teff_vals_all>=bins[i]) & (teff_vals_all<bins[i+1])]) for i in range(len(bins)-1)]
    ax.plot((bins[:-1]+bins[1:])/2, bin_meds, 'k-', lw=2, alpha=0.7)
    ax.axhline(0, color='blue', ls='--', lw=0.8, alpha=0.4)
    ax.axhline(3, color='green', ls=':', lw=1, alpha=0.5, label='z=3 threshold')
    
    rho = np.corrcoef(teff_vals_all[valid], r['_z'][valid])[0,1]
    ax.set_title(f'{cfg_name} ({r["dim"]}D) | corr(z,Teff)={rho:+.3f} | CNstar={r["cnstar_mean"]:.3f}', fontsize=10)
    ax.set_xlabel('Teff (K)'); ax.set_ylabel('Cluster z-score')
    if cfg_name == config_names[0]:
        ax.legend(fontsize=7, loc='upper right')
plt.suptitle('Cluster Z-Score vs Teff: delta_raw + Varying PCA', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ---- Visual: Trade-off curves ----
fig, axes = plt.subplots(1, 4, figsize=(22, 5))
x = np.arange(len(config_names))

# CNstar detection
ax = axes[0]
vals = [all_results[c]['cnstar_mean'] for c in config_names]
ax.bar(x, vals, color='steelblue', edgecolor='white')
for i, v in enumerate(vals):
    ax.text(i, v+0.01, f'{v:.3f}', ha='center', fontsize=9, fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(config_names, rotation=30, ha='right')
ax.set_title('CNstar Mean Prob')

# GCS detection
ax = axes[1]
vals = [all_results[c]['gcs_mean'] for c in config_names]
ax.bar(x, vals, color='darkgreen', edgecolor='white')
for i, v in enumerate(vals):
    ax.text(i, v+0.005, f'{v:.3f}', ha='center', fontsize=9, fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(config_names, rotation=30, ha='right')
ax.set_title('GCS Mean Prob')

# Avg Cohen's d
ax = axes[2]
vals = [np.mean([abs(all_results[c]['bias'][p]['d']) for p in ['teff','logg','feh']]) for c in config_names]
colors = ['forestgreen' if v < 0.3 else 'darkorange' if v < 0.5 else 'crimson' for v in vals]
ax.bar(x, vals, color=colors, edgecolor='white')
ax.axhline(0.3, color='gray', ls='--', lw=1, alpha=0.5)
for i, v in enumerate(vals):
    ax.text(i, v+0.02, f'{v:.2f}', ha='center', fontsize=9, fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(config_names, rotation=30, ha='right')
ax.set_title("Avg |Cohen's d| (lower = less bias)")

# Within-cluster correlation
ax = axes[3]
vals = [all_results[c]['wc_r_weighted'] for c in config_names]
ax.bar(x, vals, color='steelblue', edgecolor='white')
for i, v in enumerate(vals):
    ax.text(i, v+0.01, f'{v:.3f}', ha='center', fontsize=9, fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(config_names, rotation=30, ha='right')
ax.set_title('Within-Cluster Spearman r')

plt.suptitle('Detection Quality vs Bias Trade-off', fontsize=14)
plt.tight_layout()
plt.show()

## 8. Spectrum-vs-Cluster-Mean Comparison (KEY VISUALIZATION)

For the best configuration, show top candidates side-by-side with their cluster mean spectra.
This allows visual confirmation of CN band enhancement.

In [ ]:
# Select best config by composite score
scores = {}
for cfg_name in config_names:
    r = all_results[cfg_name]
    avg_d = np.mean([abs(r['bias'][p]['d']) for p in ['teff','logg','feh']])
    scores[cfg_name] = r['cnstar_mean'] * 2.0 - avg_d * 1.0 + r['wc_r_weighted'] * 1.0 + r['gcs_mean'] * 1.5
best_cfg = max(scores, key=scores.get)
print(f'Best config: {best_cfg} (score={scores[best_cfg]:.3f})')
for c, s in sorted(scores.items(), key=lambda x: -x[1]):
    print(f'  {c:15s}: {s:+.3f}')

In [ ]:
# ---- Compute CN band indices for candidate and cluster mean ----
def compute_cn_indices(wave, flux):
    """Compute CN3839, CN4142, CH4300 band indices for a single spectrum."""
    indices = {}
    for band_name in ['CN3839', 'CN4142', 'CH4300']:
        band_def = CN_BAND_DEFS[band_name]
        try:
            val = compute_band_index(wave, flux, **{k: v for k, v in band_def.items()})
        except Exception:
            val = np.nan
        indices[band_name] = val
    return indices

def get_cluster_mean_spectrum(cluster_id, dataset, cluster_labels):
    """Mean spectrum of a cluster."""
    mask = cluster_labels == cluster_id
    return dataset.X[mask].mean(axis=0)

# Get top candidates from best config
best_results = all_results[best_cfg]
top200_idx = best_results['_top200_idx']
top9_orig = unl_indices[top200_idx[:9]]  # original dataset indices
top9_clusters = neg_clusters[top200_idx[:9]]

# CN band regions for shading
cn_bands = [
    ('CN3839', 3830, 3883),
    ('CN4142', 4120, 4216),
    ('CH4300', 4285, 4315),
]

fig, axes = plt.subplots(9, 3, figsize=(20, 36),
    gridspec_kw={'width_ratios': [1, 1, 0.6]})

for i in range(9):
    idx = top9_orig[i]
    cid = top9_clusters[i]
    z_score = best_results['_z'][top200_idx[i]]
    prob = best_results['_probs'][top200_idx[i]]
    
    cand_spec = dataset.X[idx]
    cluster_spec = get_cluster_mean_spectrum(cid, dataset, cluster_labels)
    residual = cand_spec - cluster_spec
    
    cn_idx_cand = compute_cn_indices(wave, cand_spec)
    cn_idx_cluster = compute_cn_indices(wave, cluster_spec)
    
    # Column 1: Candidate spectrum
    ax = axes[i, 0]
    ax.plot(wave, cand_spec, 'k-', lw=0.6, label='Candidate')
    ax.plot(wave, cluster_spec, 'b-', lw=0.8, alpha=0.5, label='Cluster mean')
    for name, lo, hi in cn_bands:
        ax.axvspan(lo, hi, alpha=0.12, color='steelblue')
    ax.set_ylabel(f'#{i+1}')
    if i == 0:
        ax.set_title('Spectrum vs Cluster Mean', fontsize=11)
        ax.legend(fontsize=7)
    
    # Column 2: Residual (candidate - cluster)
    ax = axes[i, 1]
    ax.plot(wave, residual, 'r-', lw=0.6)
    ax.fill_between(wave, 0, residual, where=(residual<0), color='red', alpha=0.15)
    ax.fill_between(wave, 0, residual, where=(residual>0), color='blue', alpha=0.08)
    for name, lo, hi in cn_bands:
        ax.axvspan(lo, hi, alpha=0.12, color='steelblue')
    ax.axhline(0, color='gray', ls='--', lw=0.5)
    if i == 0:
        ax.set_title('Residual (Candidate - Cluster)', fontsize=11)
    
    # Column 3: CN band index comparison (bar chart)
    ax = axes[i, 2]
    bands = ['CN3839', 'CN4142', 'CH4300']
    x_pos = np.arange(len(bands))
    cand_vals = [cn_idx_cand.get(b, np.nan) for b in bands]
    clus_vals = [cn_idx_cluster.get(b, np.nan) for b in bands]
    
    w = 0.35
    bars1 = ax.bar(x_pos - w/2, cand_vals, w, color='crimson', alpha=0.8, label='Candidate')
    bars2 = ax.bar(x_pos + w/2, clus_vals, w, color='steelblue', alpha=0.8, label='Cluster')
    
    for j, (cv, lv) in enumerate(zip(cand_vals, clus_vals)):
        if not np.isnan(cv) and not np.isnan(lv):
            delta = cv - lv
            color = 'darkred' if delta < 0 else 'darkgreen'
            ax.annotate(f'{delta:+.3f}', (x_pos[j], max(cv, lv)),
                       ha='center', fontsize=7, fontweight='bold', color=color,
                       xytext=(0, 3), textcoords='offset points')
    
    ax.set_xticks(x_pos); ax.set_xticklabels(bands, fontsize=7)
    if i == 0:
        ax.set_title('CN Band Indices', fontsize=11)
        ax.legend(fontsize=7)
    
    # Annotate with z-score and prob on the residual panel
    axes[i, 0].set_title(f'#{i+1}  z={z_score:.1f}  prob={prob:.3f}  cluster={cid}',
                         fontsize=9, loc='left')

fig.suptitle(f'Top-9 Candidates: Spectrum vs Cluster Mean Comparison ({best_cfg})', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ---- Detailed view: Top 4 candidates with zoomed CN band regions ----
fig, axes = plt.subplots(4, 3, figsize=(22, 16))

for i in range(4):
    idx = top9_orig[i]
    cid = top9_clusters[i]
    cand_spec = dataset.X[idx]
    cluster_spec = get_cluster_mean_spectrum(cid, dataset, cluster_labels)
    residual = cand_spec - cluster_spec
    z_score = best_results['_z'][top200_idx[i]]
    prob = best_results['_probs'][top200_idx[i]]
    
    for j, (band_name, lo, hi) in enumerate(cn_bands):
        ax = axes[i, j]
        mask = (wave >= lo-15) & (wave <= hi+15)
        w_zoom = wave[mask]
        ax.plot(w_zoom, cand_spec[mask], 'k-', lw=1.0, label='Candidate')
        ax.plot(w_zoom, cluster_spec[mask], 'b-', lw=1.2, alpha=0.6, label='Cluster mean')
        ax.fill_between(w_zoom, cand_spec[mask], cluster_spec[mask],
                       where=(cand_spec[mask] < cluster_spec[mask]),
                       color='red', alpha=0.2, label='CN excess')
        ax.axvspan(lo, hi, alpha=0.08, color='steelblue')
        
        # Band index annotations
        cn_idx_c = compute_cn_indices(wave, cand_spec)
        cn_idx_m = compute_cn_indices(wave, cluster_spec)
        cv = cn_idx_c.get(band_name, np.nan)
        mv = cn_idx_m.get(band_name, np.nan)
        ax.set_title(f'{band_name}: cand={cv:.3f}  mean={mv:.3f}  delta={cv-mv:+.3f}', fontsize=9)
        
        if j == 1:
            ax.set_title(f'#{i+1} z={z_score:.1f} prob={prob:.3f} | '
                        f'{band_name}: cand={cv:.3f} mean={mv:.3f}', fontsize=9)
        if i == 0:
            ax.legend(fontsize=7, loc='lower left')
        if j == 0:
            ax.set_ylabel(f'#{i+1}  Norm Flux')
        if i == 3:
            ax.set_xlabel('Wavelength (A)')

fig.suptitle(f'Top-4 Candidates: CN Band Region Zoom ({best_cfg})', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ---- Statistical summary: delta_CN band index distribution ----
fig, axes = plt.subplots(3, 2, figsize=(16, 12))

for j, band in enumerate(['CN3839', 'CN4142', 'CH4300']):
    # Panel 1: Candidate delta vs cluster delta (scatter)
    ax = axes[j, 0]
    all_deltas = []
    all_delta_clusters = []
    colors_list = []
    
    for i in range(min(200, len(top200_idx))):
        idx = unl_indices[top200_idx[i]]
        cid = neg_clusters[top200_idx[i]]
        cand_spec = dataset.X[idx]
        cluster_spec = get_cluster_mean_spectrum(cid, dataset, cluster_labels)
        c_idx = compute_cn_indices(wave, cand_spec)
        m_idx = compute_cn_indices(wave, cluster_spec)
        all_deltas.append(c_idx.get(band, np.nan) - m_idx.get(band, np.nan))
        all_delta_clusters.append(c_idx.get(band, np.nan))
        colors_list.append('crimson' if (c_idx.get(band, np.nan) - m_idx.get(band, np.nan)) < 0 else 'lightgray')
    
    all_deltas = np.array(all_deltas)
    all_delta_clusters = np.array(all_delta_clusters)
    valid = ~(np.isnan(all_deltas) | np.isnan(all_delta_clusters))
    
    ax.axhline(0, color='gray', ls='--', lw=0.8)
    ax.axvline(0, color='gray', ls='--', lw=0.8)
    ax.scatter(all_delta_clusters[valid], all_deltas[valid], c=[colors_list[i] for i in range(len(valid)) if valid[i]], s=15, alpha=0.6)
    n_enhanced = (all_deltas[valid] < 0).sum()  # negative delta = CN stronger in candidate
    ax.set_title(f'{band}: {n_enhanced}/{valid.sum()} enhanced ({n_enhanced/valid.sum()*100:.1f}%)')
    ax.set_xlabel('Candidate Index'); ax.set_ylabel('Candidate - Cluster Mean')
    
    # Panel 2: Histogram of delta values
    ax = axes[j, 1]
    ax.hist(all_deltas[valid], bins=30, color='steelblue', alpha=0.7, edgecolor='white')
    ax.axvline(0, color='red', ls='--', lw=1.5, alpha=0.7, label='No enhancement')
    ax.axvline(np.median(all_deltas[valid]), color='green', ls='-', lw=1.5, alpha=0.7, label=f'Median={np.median(all_deltas[valid]):+.3f}')
    ax.set_title(f'{band}: Delta Distribution (Candidate - Cluster)')
    ax.set_xlabel('Candidate - Cluster Mean'); ax.set_ylabel('Count')
    ax.legend(fontsize=8)

fig.suptitle(f'Top-200 CN Band Enhancement Statistics ({best_cfg})', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 9. Conclusion

- **Best config**: Determined by composite score above.
- **Does adding PCA help?**: Compare delta_pca0 vs delta_pca3 vs delta_pca5+ — if CNstar improves without large bias increase, PCA adds value.
- **Are candidates real CN stars?**: Check spectrum-vs-cluster-mean comparisons — candidates should show clear absorption excess in CN3839 and CN4142 bands (blue-shaded regions showing negative residual).
- **Next step**: If a config performs well, integrate into main pipeline.